# HotWAY: Sequential Policy Routing via Pointer Network (A2C)
## Baseline Transition Model: RNN Seq2Seq vs Graph Multi-Head Attention

**Target Hardware:** Kaggle Notebook (2x NVIDIA T4 GPU)  
**Theoretical Foundation:** Vinyals et al. (NeurIPS 2015), Bello et al. (ICLR 2017)  
**Objective:** Train an Additive Attention Pointer Network with an Advantage Actor-Critic (A2C) baseline on the identical multi-scale distribution ($N \in [10, 50]$) to objectively measure the recurrent latency bottleneck against the Attention Model.

---
### Architectural Highlights:
1. **Bi-directional LSTM Encoder:** Encodes sequential dependencies and node features.
2. **Additive Attention Pointer:** Computes Bahdanau-style attention scores over encoder states to select unvisited stops sequentially.
3. **Dynamic Min-Max Scaling:** Ensures coordinates are strictly normalized to $[0, 1]^2$ per instance, eliminating regional bias across Hanoi, Da Nang, and HCMC.
4. **A2C Baseline Critic:** A parametric value network $V_\phi(s)$ predicts expected tour length to reduce policy gradient variance.


In [ ]:
# 1. Environment & Hardware Setup
import os
import sys
import math
import time
import json
import random
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Hardware: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

# Hyperparameters for Pointer Network
INPUT_DIM = 4      # [norm_lat, norm_lon, norm_dist_to_depot, congestion_weight]
HIDDEN_DIM = 128
BATCH_SIZE = 256 if torch.cuda.is_available() else 32
EPOCHS = 30
STEPS_PER_EPOCH = 400
LR = 1e-3
LR_DECAY = 0.96
MAX_NORM = 2.0
print('Pointer Network hyperparameters initialized.')


In [ ]:
# 2. Dynamic Min-Max Scaling & Traffic Helper
def dynamic_min_max_scale_coords(coords: np.ndarray) -> np.ndarray:
    min_xy = coords.min(axis=0)
    max_xy = coords.max(axis=0)
    range_xy = np.maximum(max_xy - min_xy, 1e-6)
    return (coords - min_xy) / range_xy

def compute_traffic_multiplier(hour_float: float) -> float:
    t = hour_float % 24.0
    if 7.0 <= t < 8.5:
        return 2.10
    elif 8.5 <= t < 9.0:
        return 1.50
    elif 9.0 <= t < 16.5:
        return 1.20
    elif 16.5 <= t < 17.0:
        return 1.60
    elif 17.0 <= t < 18.5:
        return 2.25
    elif 18.5 <= t < 20.0:
        return 1.40
    elif 20.0 <= t < 22.0:
        return 1.10
    else:
        return 1.00

print('Dynamic scaling and traffic helper loaded.')


In [ ]:
# 3. PyTorch Architecture: Pointer Network with A2C (Bahdanau Additive Attention)
class AttentionPointer(nn.Module):
    def __init__(self, hidden_dim: int = 128):
        super().__init__()
        self.w_enc = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.w_dec = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, encoder_outputs, decoder_state, mask=None):
        enc_proj = self.w_enc(encoder_outputs)  # (B, N, H)
        dec_proj = self.w_dec(decoder_state).unsqueeze(1)  # (B, 1, H)
        scores = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(-1)  # (B, N)
        scores = 10.0 * torch.tanh(scores / math.sqrt(encoder_outputs.size(-1)))  # Logit clipping
        if mask is not None:
            scores = scores.masked_fill(mask == 1, -1e9)
        return scores

class PointerNetwork(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dim: int = 128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Linear(input_dim, hidden_dim)
        self.encoder = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,
            batch_first=True,
            bidirectional=True
        )
        self.decoder_cell = nn.LSTMCell(input_size=hidden_dim, hidden_size=hidden_dim)
        self.pointer = AttentionPointer(hidden_dim=hidden_dim)
        self.critic_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, node_inputs, decode_type: str = 'greedy'):
        B, N, _ = node_inputs.shape
        embedded = self.embedding(node_inputs)
        enc_outputs, (h_n, c_n) = self.encoder(embedded)

        graph_repr = enc_outputs.mean(dim=1)
        baseline_pred = self.critic_head(graph_repr).squeeze(-1)

        dec_h = graph_repr
        dec_c = torch.zeros_like(dec_h)
        mask = torch.zeros(B, N, dtype=torch.uint8, device=node_inputs.device)

        # Force depot (index 0) visited first
        curr_node = torch.zeros(B, dtype=torch.long, device=node_inputs.device)
        mask[:, 0] = 1
        tours = [curr_node]
        log_probs = []

        batch_idx = torch.arange(B, device=node_inputs.device)

        for step in range(1, N):
            input_embed = embedded[batch_idx, curr_node]
            dec_h, dec_c = self.decoder_cell(input_embed, (dec_h, dec_c))
            logits = self.pointer(enc_outputs, dec_h, mask=mask)
            probs = F.softmax(logits, dim=-1)

            if decode_type == 'greedy':
                next_node = torch.argmax(probs, dim=-1)
            else:
                next_node = torch.multinomial(probs, num_samples=1).squeeze(-1)

            step_log_prob = F.log_softmax(logits, dim=-1).gather(1, next_node.unsqueeze(-1)).squeeze(-1)
            log_probs.append(step_log_prob)

            mask[batch_idx, next_node] = 1
            curr_node = next_node
            tours.append(curr_node)

        tours = torch.stack(tours, dim=1)
        total_log_prob = torch.stack(log_probs, dim=1).sum(dim=1) if log_probs else torch.zeros(B, device=node_inputs.device)
        return tours, total_log_prob, baseline_pred

ptr_test = PointerNetwork(input_dim=4, hidden_dim=128)
total_ptr_params = sum(p.numel() for p in ptr_test.parameters())
print(f'Pointer Network Initialized. Total Parameters: {total_ptr_params:,}')


In [ ]:
# 4. Multi-Scale Batch Generation & Tour Cost for Pointer Net
def generate_pointer_batch(batch_size: int, device: torch.device):
    if random.random() < 0.6:
        n_nodes = random.randint(10, 25)
    else:
        n_nodes = random.randint(26, 50)

    coords = torch.rand(batch_size, n_nodes, 2, device=device)
    traffic = 1.0 + 1.25 * torch.rand(batch_size, n_nodes, 1, device=device)
    depot_coords = coords[:, 0:1, :]
    dist_to_depot = torch.norm(coords - depot_coords, dim=-1, keepdim=True)
    features = torch.cat([coords, dist_to_depot, traffic / 2.25], dim=-1)
    return features, coords, traffic, n_nodes

def compute_tour_cost(tours: torch.Tensor, coords: torch.Tensor, traffic: torch.Tensor) -> torch.Tensor:
    B, N, _ = coords.size()
    ordered_coords = torch.gather(coords, 1, tours.unsqueeze(-1).expand(B, N, 2))
    ordered_traffic = torch.gather(traffic, 1, tours.unsqueeze(-1).expand(B, N, 1))
    seg_dists = torch.norm(ordered_coords[:, :-1, :] - ordered_coords[:, 1:, :], dim=-1)
    seg_traffic = 0.5 * (ordered_traffic[:, :-1, 0] + ordered_traffic[:, 1:, 0])
    tour_cost = (seg_dists * seg_traffic).sum(dim=-1)
    depot_dist = torch.norm(ordered_coords[:, -1, :] - ordered_coords[:, 0, :], dim=-1)
    depot_traffic = 0.5 * (ordered_traffic[:, -1, 0] + ordered_traffic[:, 0, 0])
    return tour_cost + (depot_dist * depot_traffic)

print('Batch generator & Cost function ready.')


In [ ]:
# 5. Advantage Actor-Critic (A2C) Training Loop for Pointer Network
ptr_model = PointerNetwork(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
optimizer = optim.Adam(ptr_model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=LR_DECAY)

print('=' * 85)
print(f'Starting Pointer Network A2C Training: {EPOCHS} Epochs | {STEPS_PER_EPOCH} Steps/Epoch')
print('=' * 85)

ptr_history = []
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    ptr_model.train()
    epoch_actor_losses = []
    epoch_critic_losses = []
    epoch_costs = []
    epoch_start = time.time()

    for step in range(1, STEPS_PER_EPOCH + 1):
        features, coords, traffic, n_nodes = generate_pointer_batch(BATCH_SIZE, device)

        tours, log_probs, baseline_pred = ptr_model(features, decode_type='sampling')
        cost = compute_tour_cost(tours, coords, traffic)

        # A2C Advantage
        advantage = cost.detach() - baseline_pred
        actor_loss = (advantage.detach() * log_probs).mean()
        critic_loss = F.mse_loss(baseline_pred, cost.detach())
        total_loss = actor_loss + 0.5 * critic_loss

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(ptr_model.parameters(), max_norm=MAX_NORM)
        optimizer.step()

        epoch_actor_losses.append(actor_loss.item())
        epoch_critic_losses.append(critic_loss.item())
        epoch_costs.append(cost.mean().item())

    scheduler.step()
    dur = time.time() - epoch_start
    mean_act = float(np.mean(epoch_actor_losses))
    mean_crit = float(np.mean(epoch_critic_losses))
    mean_c = float(np.mean(epoch_costs))

    print(f'Epoch {epoch:02d}/{EPOCHS:02d} ({dur:.1f}s) | Actor Loss: {mean_act:+.4f} | Critic Loss: {mean_crit:.4f} | Tour Cost: {mean_c:.4f}')
    ptr_history.append({'epoch': epoch, 'actor_loss': mean_act, 'critic_loss': mean_crit, 'tour_cost': mean_c})

print('=' * 85)
print(f'Pointer Network Training Complete in {(time.time() - start_time) / 3600.0:.2f} Hours.')
print('=' * 85)


In [ ]:
# 6. Plotting Pointer Network Convergence Curves
epochs_arr = [h['epoch'] for h in ptr_history]
costs_arr = [h['tour_cost'] for h in ptr_history]
critic_arr = [h['critic_loss'] for h in ptr_history]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Pointer Network Training Convergence (A2C)', fontsize=14, fontweight='bold')

axs[0].plot(epochs_arr, costs_arr, color='#dc2626', lw=2)
axs[0].set_title('Mean Tour Cost (Sequential LSTM Decoder)')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Cost')
axs[0].grid(True, alpha=0.3)

axs[1].plot(epochs_arr, critic_arr, color='#7c3aed', lw=2)
axs[1].set_title('Critic MSE Loss')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('MSE Loss')
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pointer_net_convergence_curves.png', dpi=200)
plt.show()
print('Saved plots to pointer_net_convergence_curves.png')


In [ ]:
# 7. Export Pointer Network Weights & Artifacts
ptr_weights = 'pointer_net_nationwide.pt'
torch.save(ptr_model.state_dict(), ptr_weights)

zip_filename = 'pointer_net_kaggle_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    zipf.write(ptr_weights)
    zipf.write('pointer_net_convergence_curves.png')

print('=' * 85)
print(f'Pointer Network packaged to: {zip_filename}')
print('Download and ingest using: python scripts/ingest_kaggle_weights.py --zip pointer_net_kaggle_artifacts.zip')
print('=' * 85)
